# Par-PARSEME: first look at the data

This notebook downloads the official data, turns it into clean CSV files and
answers three questions: which splits exist, is anything missing, and do we need
more data.

**Kaggle settings:** Internet **On**. No GPU needed. Nothing to install.
Runs in about 1 minute.

Output files (in `/kaggle/working/csv/`, also in the Output tab):

| File | Rows | What |
|---|---|---|
| `all_train.csv` | 1,601 | every training sentence, with its `fold` (0-4) |
| `train.csv` | ~1,280 | folds 1-4: what a model learns from |
| `dev.csv` | ~320 | fold 0: what we score on during development |
| `trial.csv` | 5 | organisers' trial examples (EN, FR) |
| `test.csv` | - | only once the test data is out (10 Jan 2027) |

## 1. Download the data and our shared code

In [ ]:
import os, subprocess
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("work")
os.makedirs(WORK, exist_ok=True)

REPO_URL = "https://github.com/meskkkk/Par-PARSEME-multilingual-shared-task-on-paraphrasing-of-idioms.git"
REPO = f"{WORK}/repo"
DATA = f"{WORK}/sharedtask-data"

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

# Our team repo: gives the shared data reader and the fixed fold file.
if not os.path.isdir(REPO):
    sh(f"git clone -q {REPO_URL} {REPO}")
# Organisers' data (only the par-parseme folder of their GitLab repo).
if not os.path.isdir(DATA):
    sh(f"git clone -q --depth 1 --filter=blob:none --sparse https://gitlab.com/parseme/sharedtask-data.git {DATA}")
    sh(f"git -C {DATA} sparse-checkout set par-parseme")
DATA_DIR = f"{DATA}/par-parseme"
print("Language folders:", sorted(d for d in os.listdir(DATA_DIR) if len(d) == 2))

## 2. Load everything with the shared reader
The reader (`src/shared/data.py`) cleans spacing and Unicode (no lowercasing),
and puts `[[ ]]` around the idiom inside the original sentence.

In [ ]:
import sys, json
import pandas as pd
sys.path.insert(0, REPO)
from src.shared.data import TRAIN_LANGS, load_split, load_folds
from src.shared.preprocess import unmarked_spans

train_examples = load_split(DATA_DIR, "train", TRAIN_LANGS)
trial_examples = load_split(f"{DATA_DIR}/trial", "gold")
test_examples  = load_split(DATA_DIR, "test")          # empty until 10 Jan 2027
folds = load_folds(f"{REPO}/data/folds.json")
print(f"train: {len(train_examples)}   trial: {len(trial_examples)}   test: {len(test_examples)}")

## 3. Turn them into one table
One row per sentence. Columns:

| Column | Meaning |
|---|---|
| `uid` | unique id = `lang` + `sent_id` |
| `lang` | language code (EL, FR, ...) |
| `sent_id` | organisers' `source_sent_id`, needed in the submission file |
| `fold` | 0-4, our fixed dev split (empty for trial/test) |
| `mwe_type` | VID = verbal, NID = nominal, AdjID = adjectival idiom |
| `sentence` | the input sentence (what a model receives) |
| `sentence_marked` | same sentence with the idiom in `[[ ]]` |
| `idiom` | the idiom words; pieces of a split idiom joined by ` ... ` |
| `idiom_spans` | character positions of the idiom in `sentence`, e.g. `[[12, 25]]` |
| `n_idiom_pieces` | 1 = idiom words together, 2+ = split by other words |
| `minimal` | gold rewrite that changes as little as possible |
| `creative` | gold freer rewrite |
| `organiser_text` | organisers' tokenised sentence with marks (the scorer uses it) |

In [ ]:
def to_rows(examples, split):
    rows = []
    for ex in examples:
        marked = ex.marked_text
        rows.append({
            "uid": ex.uid,
            "lang": ex.lang,
            "sent_id": ex.sent_id,
            "split": split,
            "fold": folds.get(ex.uid) if split == "train" else None,
            "mwe_type": ex.mwe_type,
            "sentence": ex.raw_text,
            "sentence_marked": marked,
            "idiom": ex.idiom,
            "idiom_spans": json.dumps([list(s) for s in unmarked_spans(marked)]) if marked else None,
            "n_idiom_pieces": marked.count("[[") if marked else None,
            "minimal": ex.minimal,
            "creative": ex.creative,
            "organiser_text": ex.original.get("text"),
        })
    return pd.DataFrame(rows)

all_train = to_rows(train_examples, "train")
all_train["fold"] = all_train["fold"].astype(int)
trial = to_rows(trial_examples, "trial")
test = to_rows(test_examples, "test")
all_train.head(3)

## 4. Train / dev split and saving the CSVs

In [ ]:
DEV_FOLD = 0                     # the team scores on all 5 folds in turn; fold 0 is the default dev set
train = all_train[all_train.fold != DEV_FOLD]
dev   = all_train[all_train.fold == DEV_FOLD]

OUT = f"{WORK}/csv"
os.makedirs(OUT, exist_ok=True)
all_train.to_csv(f"{OUT}/all_train.csv", index=False)
train.to_csv(f"{OUT}/train.csv", index=False)
dev.to_csv(f"{OUT}/dev.csv", index=False)
trial.to_csv(f"{OUT}/trial.csv", index=False)
if len(test):
    test[["uid", "lang", "sent_id", "sentence"]].to_csv(f"{OUT}/test.csv", index=False)
print({f: len(pd.read_csv(f"{OUT}/{f}")) for f in sorted(os.listdir(OUT))})

# Safety checks: no sentence is in both train and dev, and no idiom either.
assert not set(train.uid) & set(dev.uid)
key = lambda df: set(df.lang + "|" + df.idiom.str.lower())
print("idioms shared between train.csv and dev.csv:", len(key(train) & key(dev)))

## 5. Basic statistics

In [ ]:
stats = all_train.groupby("lang").agg(
    sentences=("uid", "size"),
    verbal=("mwe_type", lambda s: (s == "VID").sum()),
    nominal=("mwe_type", lambda s: (s == "NID").sum()),
    adjectival=("mwe_type", lambda s: (s == "AdjID").sum()),
    split_idioms=("n_idiom_pieces", lambda s: (s > 1).sum()),
    distinct_idioms=("idiom", lambda s: s.str.lower().nunique()),
    avg_sentence_chars=("sentence", lambda s: round(s.str.len().mean())),
)
stats.loc["TOTAL"] = stats.sum(numeric_only=True)
stats.loc["TOTAL", "avg_sentence_chars"] = round(all_train.sentence.str.len().mean())
stats

In [ ]:
print("Sentences per fold and language (dev = fold 0):")
pd.crosstab(all_train.lang, all_train.fold, margins=True)

In [ ]:
# A few examples per idiom type
for t in ["VID", "NID", "AdjID"]:
    row = all_train[(all_train.mwe_type == t) & (all_train.lang == "FR")].iloc[0]
    print(f"--- {t} ({row.lang})\n input:    {row.sentence_marked}\n minimal:  {row.minimal}\n creative: {row.creative}\n")

## 6. Is anything missing?

In [ ]:
def count(cond): return int(cond.sum())
print("Training data checks")
print("  sentences without a minimal rewrite :", count(all_train.minimal.isna()))
print("  sentences without a creative rewrite:", count(all_train.creative.isna()))
print("  (the organisers' scorer then uses the other rewrite for both)")
print("  sentences where the idiom could not be placed in the sentence:",
      count(all_train.sentence_marked.str.replace("[[", "", regex=False).str.replace("]]", "", regex=False) != all_train.sentence))
print("  duplicate ids:", count(all_train.uid.duplicated()))
print()
print("Languages in the test set without training data: EN (only 3 trial sentences), IT (none)")
print("Official dev set: none, so dev.csv is made by us from the training data")
print("Test set released:", "yes" if len(test) else "no, comes out on 10 Jan 2027 (sentences only, no idiom marks, no rewrites)")

## 7. What this means

* **Splits.** The organisers give only **train** (1,601 sentences, 13 languages)
  and a 5-sentence **trial**. There is **no official dev set**, so we made one:
  `data/folds.json` in our repo splits train into 5 folds, balanced by idiom type,
  with every idiom kept inside one fold. `dev.csv` is fold 0. For the paper we score
  all 5 folds in turn (each time training on the other 4), which gives a steadier number.
* **Test.** Comes out on 10 January 2027: about 100 sentences per language, 13
  languages plus English and Italian. It has only `sent_id` and the sentence:
  **no idiom marks and no rewrites.** That is why we need the idiom finder.
* **Do we need more data?** Not to take part: everything the task gives is here.
  The gaps are English and Italian (no training data) and the small size per
  language (66-150 sentences). Our plan covers them by borrowing examples across
  languages (RAG). Any extra outside data must be allowed by the task rules and
  cited, so check the official website before adding some.